# 01.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/03-pandas-basics/solutions.ipynb)

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Invoice":     [536365, 536365, 536365, 536366, 536367],
    "StockCode":   ["85123A", "71053", "84406B", "22633", "84879"],
    "Quantity":    [6, 6, np.nan, 6, 32],
    "Price":       [2.55, 3.39, 2.75, 1.85, 1.69],
    "Customer ID": [17850, 17850, 17850, np.nan, 13047],
    "Country":     ["United Kingdom"] * 4 + ["France"],
})
df

,Invoice,StockCode,Quantity,Price,Customer ID,Country
0,536365,85123A,6.0,2.55,17850.0,United Kingdom
1,536365,71053,6.0,3.39,17850.0,United Kingdom
2,536365,84406B,NaN,2.75,17850.0,United Kingdom
3,536366,22633,6.0,1.85,NaN,United Kingdom
4,536367,84879,32.0,1.69,13047.0,France


## Exercise 1

Prices: 2.55, 3.39, 2.75, 1.85, 1.69. Count 5. Sum = 12.23, mean = 2.446. Min 1.69, max 3.39. Sorted: 1.69, 1.85, 2.55, 2.75, 3.39 → median (middle of five) = **2.55**.

In [2]:
df["Price"].describe().round(3)

count    5.000
mean     2.446
std      0.693
min      1.690
25%      1.850
50%      2.550
75%      2.750
max      3.390
Name: Price, dtype: float64

## Exercise 2

In [3]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

cust = pd.read_csv(data_path("shopwave_customers_small.csv"))
num = cust.select_dtypes("number")
(num.std() / num.mean()).round(2).sort_values(ascending=False)

churned              1.13
days_since_last      0.52
visits_last_month    0.49
spend_next_month     0.44
items_last_month     0.42
dtype: float64

`spend_next_month` and `visits_last_month` vary most relative to their size (and `churned`, being 0/1, is a special case). Customers differ a lot in how much they spend; that spread is what a regression model will try to explain.

## Exercise 3

In [4]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, time, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")

if csv.exists():
    retail = pd.read_csv(csv, parse_dates=["InvoiceDate"])
else:
    t0 = time.perf_counter()
    retail = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    print(f"read_excel took {time.perf_counter() - t0:.0f} s")
    retail.to_csv(csv, index=False)            # save a fast copy for later lessons
print(retail.shape)

(541910, 8)


In [5]:
print("missing Customer ID :", retail["Customer ID"].isna().sum())
print("distinct StockCode  :", retail["StockCode"].nunique())
print("distinct Description:", retail["Description"].nunique())

missing Customer ID : 135080
distinct StockCode  : 4070
distinct Description: 4223


There are more descriptions than stock codes: the same product has been entered with slightly different names (typos, capitalisation, added notes like "damaged"). Text keys are messy; numeric codes are what you should group by.

## Exercise 4

In [6]:
retail.describe(include=["object", "str"])

,Invoice,StockCode,Description,Country
count,541910,541910,540456,541910
unique,25900,4070,4223,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,1114,2313,2369,495478


The `top` row of `Description` is the most common product name and `freq` is how many lines it appears on.